In [2]:
from agents import Agent, Runner, function_tool, ItemHelpers
from dotenv import load_dotenv

load_dotenv()


@function_tool
def get_weather(city: str):
    """Get weather by city"""
    return "30 degrees"


agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather],
)

stream = Runner.run_streamed(
    agent, "Hello how are you? What is the weather in the capital of Spain?"
)

async for event in stream.stream_events():

    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print("Agent updated to", event.new_agent.name)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict())
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item))
    print("=" * 20)

Agent updated to Assistant Agent
{'arguments': '{"city":"Madrid"}', 'call_id': 'call_W7GauNI6dSkitPAUKZ1PL0Es', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_010e2108d1d6a225006a2937693d20819aada482c13c9899fc', 'status': 'completed'}
30 degrees


OPENAI_API_KEY is not set, skipping trace export


I’m doing well, thanks! In Madrid, the capital of Spain, it’s 30° right now.


OPENAI_API_KEY is not set, skipping trace export


In [3]:
from agents import Agent, Runner, function_tool, ItemHelpers
from dotenv import load_dotenv

load_dotenv()


@function_tool
def get_weather(city: str):
    """Get weather by city"""
    return "30 degrees"


agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather],
)

stream = Runner.run_streamed(
    agent, "Hello how are you? What is the weather in the capital of Spain?"
)

message = ""
args = ""

async for event in stream.stream_events():

    if event.type == "raw_response_event":
        event_type = event.data.type
        if event_type == "response.output_text.delta":
            message += event.data.delta
            print(message)
        elif event_type == "response.function_call_arguments.delta":
            args += event.data.delta
            print(args)
        elif event_type == "response.completed":
            message = ""
            args = ""

{"
{"city
{"city":"
{"city":"Madrid
{"city":"Madrid"}
I
I’m
I’m doing
I’m doing well
I’m doing well,
I’m doing well, thanks
I’m doing well, thanks!


I’m doing well, thanks!

The
I’m doing well, thanks!

The weather
I’m doing well, thanks!

The weather in
I’m doing well, thanks!

The weather in Madrid
I’m doing well, thanks!

The weather in Madrid,
I’m doing well, thanks!

The weather in Madrid, the
I’m doing well, thanks!

The weather in Madrid, the capital
I’m doing well, thanks!

The weather in Madrid, the capital of
I’m doing well, thanks!

The weather in Madrid, the capital of Spain
I’m doing well, thanks!

The weather in Madrid, the capital of Spain,
I’m doing well, thanks!

The weather in Madrid, the capital of Spain, is
I’m doing well, thanks!

The weather in Madrid, the capital of Spain, is 
I’m doing well, thanks!

The weather in Madrid, the capital of Spain, is 30
I’m doing well, thanks!

The weather in Madrid, the capital of Spain, is 30 degrees
I’m doing well, thanks!

The

OPENAI_API_KEY is not set, skipping trace export


In [ ]:
from agents import Agent, Runner, function_tool, ItemHelpers, SQLiteSession
from dotenv import load_dotenv

load_dotenv()

session = SQLiteSession("user_1", "ai-memory.db")


@function_tool
def get_weather(city: str):
    """Get weather by city"""
    return "30 degrees"


agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather],
)

result = await Runner.run(
    agent,
    "Hello how are you? What is the weather in the capital of Spain?",
    session=session,
)

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


I’m good, thanks! The weather in Madrid is 30 degrees.


OPENAI_API_KEY is not set, skipping trace export


In [10]:
from agents import Agent, Runner, ItemHelpers, SQLiteSession, function_tool, trace
from agents.extensions.visualization import draw_graph
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

session = SQLiteSession("user_2", "ai-memory.db")


class Answer(BaseModel):
    answer: str
    background_explanation: str


@function_tool
def get_weather():
    """Get weather"""
    return "30 degrees"


geography_agent = Agent(
    name="Geo_Expert_Agent",
    instructions="You are a expert in geography, you answer questions related to them.",
    handoff_description="Use this to answer geography related questions.",
    tools=[
        get_weather,
    ],
    output_type=Answer,
)

economics_agent = Agent(
    name="Economics_Expert_Agent",
    instructions="You are a expert in economics, you answer questions related to them.",
    handoff_description="Use this to answer economics related questions.",
)

main_agent = Agent(
    name="Assistant_Agent",
    instructions="You are a user facing agent. Transfer to the agent most capable of answering the user's question.",
    handoffs=[
        economics_agent,
        geography_agent,
    ],
)

draw_graph(main_agent)

with trace("user_2"):
    result = await Runner.run(
        main_agent,
        "What is the capital of Thailand's northen province.",
        session=session,
    )
    result = await Runner.run(
        main_agent,
        "What is the capital of Canada's northen province.",
        session=session,
    )

print(result.last_agent.name)
print(result.final_output)

Assistant_Agent
Canada doesn’t have a northern province. If you mean **Yukon**, its capital is **Whitehorse**.


OPENAI_API_KEY is not set, skipping trace export
